# Ejercicio 5 — Programación Orientada a Objetos
## Registro Climático de Guatemala — Examen Final (Variante B)

**Nombre:** ___________________________  
**Carnet:** ___________________________

---

Este notebook usa las clases que implementaste en `poo_clima.py`.

**Requisito previo:** haber completado todos los métodos de `poo_clima.py`.

**Entrega:**
- `poo_clima.py` con tu implementación
- Este notebook ejecutado con todos los outputs visibles

> **Ruta esperada en tu repositorio:**  
> `examenes/python/examen_final/poo_clima.py`  
> `examenes/python/examen_final/examen_final_poo_b.ipynb`

In [ ]:
import pandas as pd
import requests

URL_CLIMA = (
    "https://archive-api.open-meteo.com/v1/archive"
    "?latitude=14.64&longitude=-90.51"
    "&start_date=2024-01-01&end_date=2024-12-31"
    "&daily=temperature_2m_max,temperature_2m_min"
    ",precipitation_sum,wind_speed_10m_max"
    "&timezone=America%2FGuatemala"
)

respuesta = requests.get(URL_CLIMA, timeout=15)
clima = pd.DataFrame(respuesta.json()['daily'])
clima['time'] = pd.to_datetime(clima['time'])
print(f"Dataset cargado: {len(clima)} días")
clima.head(3)

In [ ]:
# Reutiliza la función clasificar_dia() del Ejercicio 3 para agregar la columna 'tipo_dia'
def clasificar_dia(precip):
    if precip < 1:
        return 'Seco'
    elif precip < 5:
        return 'Lluvia ligera'
    elif precip < 20:
        return 'Lluvia moderada'
    else:
        return 'Lluvia intensa'

clima['tipo_dia'] = clima['precipitation_sum'].apply(clasificar_dia)
print("Columna 'tipo_dia' agregada.")
print(clima['tipo_dia'].value_counts())

In [ ]:
from poo_clima import DiaClimatico, RegistroAnual
print("Clases importadas correctamente.")

---
## Parte 1 — Probando la clase `DiaClimatico` (objetos individuales)

El siguiente bloque crea tres objetos `DiaClimatico` representativos y llama a todos sus métodos.  
Verifica que el output tiene sentido con los datos climáticos.

In [ ]:
# Tres días representativos del DataFrame
fila_primero  = clima.iloc[0]
fila_caluroso = clima.loc[clima['temperature_2m_max'].idxmax()]
fila_lluvioso = clima.loc[clima['precipitation_sum'].idxmax()]

def crear_dia(fila):
    return DiaClimatico(
        fila['time'].date(),
        fila['temperature_2m_max'],
        fila['temperature_2m_min'],
        fila['precipitation_sum'],
        fila['wind_speed_10m_max'],
    )

d_primero  = crear_dia(fila_primero)
d_caluroso = crear_dia(fila_caluroso)
d_lluvioso = crear_dia(fila_lluvioso)

for etiqueta, d in [("Primero del año",       d_primero),
                    ("Día más caluroso",       d_caluroso),
                    ("Día más lluvioso",       d_lluvioso)]:
    print(f"=== {etiqueta} ===")
    print(f"  {d}")
    print(f"  clasificar()   → {d.clasificar()}")
    print(f"  rango_termico()→ {d.rango_termico():.1f} °C")
    print(f"  temp_media()   → {d.temp_media():.1f} °C")
    print(f"  es_caluroso()  → {d.es_caluroso()}")
    print()

### Tu turno — elige un día

Crea un objeto `DiaClimatico` usando **cualquier fila** del DataFrame que tú elijas  
(puede ser el día de tu cumpleaños, el más frío, el más ventoso, etc.).  
Muestra su `descripcion()` y justifica tu elección en un comentario.

In [ ]:
# Elige una fila y crea tu propio objeto DiaClimatico
# Hint: clima[clima['time'].dt.month == 8].iloc[0]  → primer día de agosto
#       clima.loc[clima['wind_speed_10m_max'].idxmax()]  → día más ventoso

# TU CÓDIGO AQUÍ
mi_fila = ...
mi_dia  = DiaClimatico(...)

print(mi_dia)
print(f"¿Es caluroso? {mi_dia.es_caluroso()}")
print(f"Rango térmico: {mi_dia.rango_termico():.1f} °C")

---
## Parte 2 — Construyendo el `RegistroAnual` con todos los datos

Crea el registro con los **366 días** del año 2024 y ejerce los métodos de consulta.

In [ ]:
registro = RegistroAnual("Ciudad de Guatemala", 2024)

for indice, fila in clima.iterrows():
    registro.agregar_dia(DiaClimatico(
        fecha         = fila['time'].date(),
        temp_max      = fila['temperature_2m_max'],
        temp_min      = fila['temperature_2m_min'],
        precipitacion = fila['precipitation_sum'],
        viento_max    = fila['wind_speed_10m_max'],
    ))

print(f"Registro construido: {len(registro)} días")

In [ ]:
# ── dia_mas_caluroso() ───────────────────────────────────────────────────────
mas_caluroso = registro.dia_mas_caluroso()
print("Día más caluroso del año:")
print(" ", mas_caluroso)
print(f"  Rango térmico : {mas_caluroso.rango_termico():.1f} °C")
print(f"  Temperatura media: {mas_caluroso.temp_media():.1f} °C")

In [ ]:
# ── dias_por_tipo() ──────────────────────────────────────────────────────────
print("Días por tipo (año completo):")
for tipo in ['Seco', 'Lluvia ligera', 'Lluvia moderada', 'Lluvia intensa']:
    grupo = registro.dias_por_tipo(tipo)
    print(f"  {tipo:<18}: {len(grupo)} días")
    for d in grupo[:2]:
        print(f"      {d}")

In [ ]:
# ── temp_promedio_anual() + resumen() ────────────────────────────────────────
print(f"Temperatura máxima promedio anual: {registro.temp_promedio_anual()} °C\n")
registro.resumen()

### Tu turno — días calurosos y lluviosos

Usa `dias_por_tipo('Lluvia intensa')` para obtener los días con lluvia intensa  
y determina cuántos de ellos también fueron calurosos (`es_caluroso() == True`).  
Muestra el porcentaje.

In [ ]:
# Hint: obtén la lista de días de lluvia intensa con dias_por_tipo()
#       luego cuenta cuántos tienen es_caluroso() == True con un for

# TU CÓDIGO AQUÍ

---
## Parte 3 — Verificación automática

Esta celda confirma que tu `clasificar()` produce los mismos resultados  
que la función `clasificar_dia()` del Ejercicio 3 para **todos** los días.

In [ ]:
errores = 0
for _, fila in clima.iterrows():
    d = DiaClimatico(fila['time'].date(), fila['temperature_2m_max'],
                     fila['temperature_2m_min'], fila['precipitation_sum'],
                     fila['wind_speed_10m_max'])
    if d.clasificar() != fila['tipo_dia']:
        errores += 1
        print(f"  ✗ {fila['time'].date()}  precip={fila['precipitation_sum']:.1f}"
              f"  pandas={fila['tipo_dia']}  poo={d.clasificar()}")

if errores == 0:
    print(f"✓ Todos los {len(clima)} días clasifican correctamente.")
else:
    print(f"✗ {errores} días con clasificación incorrecta. Revisa tu método clasificar().")

---
## Preguntas de interpretación

1. ¿En qué fecha fue el día más caluroso del año? ¿En qué mes cae?  
   ¿Coincide con lo que esperarías del clima de Guatemala? Justifica.

   *Tu respuesta:* ___

2. ¿Cuántos días de `'Lluvia intensa'` también fueron calurosos (`es_caluroso() == True`)?  
   ¿Qué te dice eso sobre la relación entre temperatura y lluvia intensa en Guatemala?

   *Tu respuesta:* ___

3. ¿Qué ventaja tiene usar la clase `RegistroAnual` frente a trabajar directamente  
   con el DataFrame de pandas para este tipo de consultas?

   *Tu respuesta:* ___